# Medallion Data Warehouse: Build, Validate, and Analyze

This tutorial builds the repository's SQL Server warehouse, runs its quality checks, and queries the Gold star schema.

**Audience:** learners familiar with SQL, Python, and basic dimensional modelling.

**Prerequisites:** Docker SQL Server started from `compose.yaml`, Microsoft ODBC Driver 18, the packages in `requirements.txt`, and `MSSQL_SA_PASSWORD` exported in the notebook process.

**Learning goals**

1. Execute the Bronze → Silver → Gold build in dependency order.
2. Fail fast when a SQL batch or stored procedure fails.
3. Interpret machine-readable Silver and Gold quality checks.
4. Calculate traceable sales KPIs and trend summaries from the Gold views.


## 1. Imports and configuration

Connection settings come from environment variables; no password or machine-specific driver path is stored in the notebook. Run the notebook from the repository root.


In [ ]:
from pathlib import Path
import os
import re

import matplotlib.pyplot as plt
import pandas as pd
import pyodbc
import seaborn as sns

ROOT = Path.cwd()
SQL_ROOT = ROOT / "warehouse_queries"
if not SQL_ROOT.exists():
    raise RuntimeError("Run this notebook from the repository root.")

SERVER = os.getenv("MSSQL_SERVER", "localhost,1433")
USERNAME = os.getenv("MSSQL_USER", "sa")
PASSWORD = os.getenv("MSSQL_SA_PASSWORD")
DRIVER = os.getenv("MSSQL_ODBC_DRIVER", "ODBC Driver 18 for SQL Server")

if not PASSWORD:
    raise RuntimeError("Export MSSQL_SA_PASSWORD before running the notebook.")

def get_connection(database="master", autocommit=False):
    connection = pyodbc.connect(
        f"DRIVER={{{DRIVER}}};"
        f"SERVER={SERVER};"
        f"DATABASE={database};"
        f"UID={USERNAME};"
        f"PWD={PASSWORD};"
        "Encrypt=yes;TrustServerCertificate=yes;",
        autocommit=autocommit,
    )
    return connection

sns.set_theme(style="whitegrid")


## 2. SQL execution helpers

`GO` is a client-side batch separator, not T-SQL. The helper splits only lines containing `GO`, raises the original database error, and rolls back non-autocommit work instead of printing a false success message.


In [ ]:
GO_PATTERN = re.compile(r"^\s*GO\s*;?\s*$", flags=re.IGNORECASE | re.MULTILINE)

def split_sql_batches(sql_text):
    return [batch.strip() for batch in GO_PATTERN.split(sql_text) if batch.strip()]

def execute_sql_file(relative_path, database="data_warehouse_prj", autocommit=False):
    path = ROOT / relative_path
    batches = split_sql_batches(path.read_text(encoding="utf-8"))
    connection = get_connection(database, autocommit=autocommit)
    cursor = connection.cursor()
    try:
        for batch_number, batch in enumerate(batches, start=1):
            cursor.execute(batch)
            while cursor.nextset():
                pass
        if not autocommit:
            connection.commit()
    except Exception as exc:
        if not autocommit:
            connection.rollback()
        raise RuntimeError(f"{path}: SQL batch {batch_number} failed") from exc
    finally:
        cursor.close()
        connection.close()

def execute_procedure(procedure_name):
    connection = get_connection("data_warehouse_prj")
    cursor = connection.cursor()
    try:
        cursor.execute(f"EXEC {procedure_name}")
        while cursor.nextset():
            pass
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        cursor.close()
        connection.close()

def query_dataframe(sql, database="data_warehouse_prj"):
    connection = get_connection(database)
    cursor = connection.cursor()
    try:
        cursor.execute(sql)
        while cursor.description is None:
            if not cursor.nextset():
                return pd.DataFrame()
        columns = [column[0] for column in cursor.description]
        return pd.DataFrame.from_records(cursor.fetchall(), columns=columns)
    finally:
        cursor.close()
        connection.close()


## 3. Build the warehouse

> **Destructive development step:** `architecture.sql` drops and recreates `data_warehouse_prj`. Do not point this notebook at a database containing data you need to retain.


In [ ]:
execute_sql_file(
    "warehouse_queries/architecture.sql",
    database="master",
    autocommit=True,
)

definition_files = [
    "warehouse_queries/bronze/ddl_bronze.sql",
    "warehouse_queries/silver/ddl_silver.sql",
    "warehouse_queries/bronze/load_bronze.sql",
    "warehouse_queries/silver/proc_load_silver.sql",
    "warehouse_queries/gold/ddl_gold.sql",
    "warehouse_queries/Inspect/test_silver.sql",
    "warehouse_queries/Inspect/test_gold.sql",
]

for sql_file in definition_files:
    print(f"Defining {sql_file}")
    execute_sql_file(sql_file)

pipeline_procedures = [
    "sp_create_bronze_tables",
    "sp_create_silver_tables",
    "bronze.load_bronze",
    "silver.load_silver",
    "sp_create_gold_views",
]

for procedure in pipeline_procedures:
    print(f"Running {procedure}")
    execute_procedure(procedure)

print("Warehouse build completed.")


## 4. Validate Silver and Gold

The procedures return one result set with `check_name`, `severity`, and `failed_rows`. Warnings preserve known source incompleteness; non-zero errors indicate that a warehouse invariant is broken.


In [ ]:
silver_checks = query_dataframe("EXEC sp_test_silver")
gold_checks = query_dataframe("EXEC sp_test_gold")

display(silver_checks)
display(gold_checks)

quality_errors = pd.concat(
    [silver_checks.assign(layer="silver"), gold_checks.assign(layer="gold")],
    ignore_index=True,
).query("severity == 'error' and failed_rows > 0")

if quality_errors.empty:
    print("No error-severity quality checks failed.")
else:
    raise AssertionError(f"Warehouse quality errors:\n{quality_errors.to_string(index=False)}")


## 5. Gold model profile and KPIs

The Gold layer is a virtual star schema: two dimension views and one fact view. The record grain is one row per customer, one row per active product, and one row per sales line, respectively.


In [ ]:
row_counts = query_dataframe("""
SELECT 'gold.dim_customers' AS object_name, COUNT_BIG(*) AS row_count FROM gold.dim_customers
UNION ALL
SELECT 'gold.dim_products', COUNT_BIG(*) FROM gold.dim_products
UNION ALL
SELECT 'gold.fact_sales', COUNT_BIG(*) FROM gold.fact_sales;
""")

kpis = query_dataframe("""
SELECT
    SUM(CAST(sales_amount AS BIGINT)) AS total_revenue,
    COUNT(DISTINCT order_number) AS distinct_orders,
    COUNT(DISTINCT customer_key) AS distinct_customers,
    CAST(
        SUM(CAST(sales_amount AS DECIMAL(19, 2)))
        / NULLIF(COUNT(DISTINCT order_number), 0)
        AS DECIMAL(19, 2)
    ) AS average_order_value
FROM gold.fact_sales;
""")

display(row_counts)
display(kpis)


## 6. Monthly revenue and business breakdowns


In [ ]:
monthly_revenue = query_dataframe("""
SELECT
    DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1) AS order_month,
    SUM(CAST(sales_amount AS BIGINT)) AS revenue,
    COUNT(DISTINCT order_number) AS orders,
    COUNT(DISTINCT customer_key) AS active_customers
FROM gold.fact_sales
WHERE order_date IS NOT NULL
GROUP BY DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1)
ORDER BY order_month;
""")
monthly_revenue["order_month"] = pd.to_datetime(monthly_revenue["order_month"])

country_revenue = query_dataframe("""
SELECT c.country, SUM(CAST(f.sales_amount AS BIGINT)) AS revenue
FROM gold.fact_sales AS f
JOIN gold.dim_customers AS c ON f.customer_key = c.customer_key
GROUP BY c.country
ORDER BY revenue DESC;
""")

product_line_revenue = query_dataframe("""
SELECT p.product_line, SUM(CAST(f.sales_amount AS BIGINT)) AS revenue
FROM gold.fact_sales AS f
JOIN gold.dim_products AS p ON f.product_key = p.product_key
GROUP BY p.product_line
ORDER BY revenue DESC;
""")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(monthly_revenue["order_month"], monthly_revenue["revenue"], color="#1F4E79")
axes[0].set_title("Monthly revenue")
axes[0].set_ylabel("Revenue")

sns.barplot(
    data=country_revenue,
    x="revenue",
    y="country",
    color="#D9A441",
    ax=axes[1],
)
axes[1].set_title("Revenue by customer country")
axes[1].set_xlabel("Revenue")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

display(product_line_revenue)


## Interpretation and limitations

- The source contains 19 sales lines whose order dates cannot be parsed; they remain in the fact view with a null `order_date` and are excluded from monthly trends.
- `CO_PE` in CRM is explicitly harmonized to the ERP category ID `CO_PD` for Components / Pedals.
- Gold dimensions are views whose `ROW_NUMBER()` surrogate keys are reproducible for this snapshot but are not durable keys for an incrementally historized warehouse.
- The first and last months in the trend are partial periods. Do not interpret their totals as comparable full months.
- This is a snapshot-oriented educational warehouse; it does not implement orchestration, slowly changing dimensions, incremental ingestion, or production observability.


## Exercise

Calculate revenue and unit volume by product category. Confirm that no fact rows have a missing category after the `CO_PE` → `CO_PD` harmonization.


In [ ]:
RUN_EXERCISE_SOLUTION = False

if RUN_EXERCISE_SOLUTION:
    category_performance = query_dataframe("""
    SELECT
        p.category,
        SUM(CAST(f.sales_amount AS BIGINT)) AS revenue,
        SUM(CAST(f.quantity AS BIGINT)) AS units
    FROM gold.fact_sales AS f
    JOIN gold.dim_products AS p ON f.product_key = p.product_key
    GROUP BY p.category
    ORDER BY revenue DESC;
    """)
    display(category_performance)
else:
    print("Set RUN_EXERCISE_SOLUTION = True after writing your query.")


## Next extensions

1. Replace view-generated surrogate keys with persisted dimension keys.
2. Add a date dimension and explicit fiscal-period logic.
3. Introduce incremental loads, orchestration, and run-level audit tables.
4. Convert the SQL checks into CI tests against an ephemeral SQL Server container.
